# Transformación de datos {#sec-data-transform}

## Introducción

Es muy raro que los datos lleguen exactamente en la forma que necesitas. A menudo tendrás que crear algunas variables o resúmenes nuevos, o quizá solo quieras renombrar las variables o reordenar las observaciones para que sea un poco más fácil trabajar con los datos.

En este capítulo aprenderás a hacer todo eso (¡y más!). Te presentaremos la transformación de datos con el paquete **pandas** y un nuevo dataset sobre los vuelos que salieron de la ciudad de Nueva York en 2013.

El objetivo de este capítulo es darte una visión general de todas las herramientas clave para transformar un dataframe, un tipo especial de objeto que contiene datos tabulares.

Volveremos a estas funciones con más detalle en capítulos posteriores, cuando empecemos a profundizar en tipos de datos específicos (p. ej., números, strings, fechas).

### Requisitos previos

En este capítulo nos centraremos en el paquete **pandas**, una de las herramientas más utilizadas en ciencia de datos. Tendrás que asegurarte de tener **pandas** instalado. Para ello, puedes ejecutar

In [ ]:
import pandas as pd

Si este comando falla, no tienes **pandas** instalado. Abre la terminal en Visual Studio Code (Terminal -> New Terminal), usa `cd` para ir a la carpeta en la que estás trabajando y escribe `uv add pandas`.

Además, si quieres comprobar qué versión de **pandas** estás usando, se hace así

In [ ]:
pd.__version__

También necesitarás los datos. La mayoría de las veces, los datos tendrán que cargarse desde un archivo o desde internet. Estos datos no son una excepción, pero una de las cosas increíbles de **pandas** es la cantidad de tipos de datos distintos que puede cargar, incluso desde archivos en internet.

Los datos ocupan unos 50 MB, así que necesitarás una buena conexión a internet o un poco de paciencia para que se descarguen.

Descarguemos los datos:

In [ ]:
url = "https://raw.githubusercontent.com/byuidatascience/data4python4ds/master/data-raw/flights/flights.csv"
flights = pd.read_csv(url)

Si el código anterior funcionó, habrás descargado los datos en formato CSV y los habrás guardado en un dataframe. Veamos las primeras filas con la función `.head()`, que funciona con todos los dataframes de **pandas**.

In [ ]:
flights.head()

Para obtener información más general sobre las columnas, los tipos de datos (`dtypes`) de las columnas y el tamaño del dataset, usa `.info()`.

In [ ]:
flights.info()

Quizá hayas notado las abreviaturas cortas que aparecen en la columna `Dtypes`. Indican el tipo de los valores de sus respectivas columnas: `int64` es la abreviatura de entero (es decir, números enteros) y `float64` es la abreviatura de número de punto flotante de doble precisión (números reales). `object` es una especie de categoría comodín para cualquier tipo de dato que **pandas** no se atreve a inferir con seguridad. Aunque no aparecen aquí, otros tipos de datos son `string` para texto y `datetime` para combinaciones de fecha y hora.

La siguiente tabla muestra algunos de los tipos de datos más comunes que probablemente encontrarás.

|  **Nombre del tipo de dato**  |    **Tipo de dato**   |
|:----------:|:-------------:|
|   float64  |  números reales |
|  category  |   categorías  |
| datetime64 |   fechas y horas  |
|    int64   |    enteros   |
|    bool    | True o False |
|   string   |      texto     |

Los distintos tipos de datos de las columnas son importantes porque las operaciones que puedes realizar en una columna dependen mucho de su "tipo"; por ejemplo, puedes eliminar todos los signos de puntuación de los strings, mientras que puedes multiplicar ints y floats.

Nos gustaría trabajar con la variable `"time_hour"` en forma de datetime; por suerte, **pandas** facilita realizar esa conversión en esa columna concreta

In [ ]:
flights["time_hour"]

In [ ]:
flights["time_hour"] = pd.to_datetime(flights["time_hour"], format="%Y-%m-%dT%H:%M:%SZ")

## Conceptos básicos de **pandas**

**pandas** es un paquete realmente completo, y este libro apenas rozará la superficie de lo que puede hacer. Pero está construido en torno a unas pocas ideas sencillas que, una vez que las entiendes, te facilitan mucho la vida.

Empecemos por lo más básico. El objeto más básico de pandas es el DataFrame. Un DataFrame es una estructura de datos bidimensional que puede almacenar datos de distintos tipos (incluidos caracteres, enteros, valores de punto flotante, datos categóricos e incluso listas) en columnas. Está formado por filas y columnas (cada celda fila-columna contiene un valor), más dos elementos de información contextual: el índice (que contiene información sobre cada fila) y los nombres de las columnas (que contienen información sobre cada columna).

![](https://pandas.pydata.org/docs/_images/01_table_dataframe.svg)

Quizá la noción más importante sobre los dataframes de **pandas** es que están construidos en torno a un índice situado en el lado izquierdo del dataframe. Cada vez que realizas una operación sobre un dataframe, tienes que pensar en cómo podría afectar o no al índice; o, dicho de otro modo, si quieres modificar el índice.

Veamos un ejemplo sencillo de esto con un dataframe inventado:

In [ ]:
df = pd.DataFrame(
    data={
        "col0": [0, 0, 0, 0],
        "col1": [0, 0, 0, 0],
        "col2": [0, 0, 0, 0],
        "col3": ["a", "b", "b", "a"],
        "col4": ["alpha", "gamma", "gamma", "gamma"],
    },
    index=["row" + str(i) for i in range(4)],
)
df.head()

Puedes ver que hay 5 columnas (llamadas de `"col0"` a `"col4"`) y que el índice consta de cuatro entradas llamadas de `"row0"` a `"row3"`.

Un segundo punto clave que debes conocer es que las operaciones sobre un dataframe de **pandas** se pueden encadenar. No necesitamos hacer una asignación por línea de código; de hecho, podemos hacer varias asignaciones en un solo comando.

Veamos un ejemplo. Vamos a encadenar cuatro operaciones:

1. usaremos `query()` para encontrar solo las filas en las que la columna de destino `"dest"` tiene el valor `"IAH"`. Esto no cambia el índice, solo elimina las filas irrelevantes. En efecto, este paso elimina las filas que no nos interesan.
2. usaremos `groupby()` para agrupar las filas por año, mes y día (pasamos una lista de columnas a la función `groupby()`). Este paso cambia el índice; el nuevo índice tendrá tres columnas que registran el año, el mes y el día. En efecto, este paso cambia el índice.
3. elegiremos qué columnas queremos conservar después de la operación `groupby()` pasando una lista de ellas dentro de unos corchetes (los corchetes dobles se deben a que es una lista dentro de un dataframe). Aquí solo queremos una columna, `"arr_delay"`. Esto no afecta al índice. En efecto, este paso elimina las columnas que no nos interesan.
4. por último, debemos especificar qué operación de `groupby()` queremos aplicar; al agregar la información de varias filas en una sola, tenemos que indicar cómo debe agregarse esa información. En este caso, usaremos `mean()`. En efecto, este paso aplica un estadístico a la(s) variable(s) que seleccionamos antes, en los grupos que creamos antes.

In [ ]:
(flights.query("dest == 'IAH'").groupby(["year", "month", "day"])[["arr_delay"]].mean())

Aquí puedes ver que hemos creado un nuevo dataframe con un nuevo índice. Para ello, usamos cuatro operaciones clave:

1. manipular filas
2. manipular el índice
3. manipular columnas
4. aplicar estadísticos

Estas cubren la mayoría de las operaciones que podrías querer hacer sobre un único dataframe, pero hay distintas opciones para cada una según lo que necesites.

Profundicemos ahora un poco más en estas operaciones.

## Manipulación de filas en dataframes

Creemos algunos datos ficticios para mostrar cómo funciona esto.

In [ ]:
import numpy as np

df = pd.DataFrame(
    data=np.reshape(range(36), (6, 6)),
    index=["a", "b", "c", "d", "e", "f"],
    columns=["col" + str(i) for i in range(6)],
    dtype=float,
)
df["col6"] = ["apple", "orange", "pineapple", "mango", "kiwi", "lemon"]
df

### Acceso a filas

Para acceder directamente a una fila concreta, puedes usar `df.loc['rowname']`, o `df.loc[['rowname1', 'rowname2']]` para dos filas distintas.

Por ejemplo,

In [ ]:
df.loc[["a", "b"]]

Pero también puedes acceder a filas concretas según su posición en el dataframe usando `.iloc`. Recuerda que los índices en Python empiezan en cero, así que para obtener la primera fila usarías `.iloc[0]`:


In [ ]:
df.iloc[0]

Esto también funciona con varias filas. Obtengamos la primera y la tercera fila (en las posiciones 0 y 2) pasando una lista de posiciones:

In [ ]:
df.iloc[[0, 2]]

Hay otras formas de acceder a varias filas que utilizan el *slicing* (rebanado), pero dejaremos ese tema para otro momento.

### Filtrar filas con query

Como en el ejemplo de los vuelos, también podemos filtrar filas según una condición usando `query()`:

In [ ]:
df.query("col6 == 'kiwi' or col6 == 'pineapple'")

Para números, también puedes usar los signos de mayor que y menor que:

In [ ]:
df.query("col0 > 6")

De hecho, hay muchas opciones que funcionan con `query()`: además de `>` (mayor que), puedes usar `>=` (mayor o igual que), `<` (menor que), `<=` (menor o igual que), `==` (igual a) y `!=` (distinto de). También puedes usar los comandos `and` y `or` para combinar varias condiciones. Aquí tienes un ejemplo de `and` con el dataframe `flights`:

In [ ]:
# Flights that departed on January 1
flights.query("month == 1 and day == 1")

Ten en cuenta que la igualdad se comprueba con `==` y *no* con `=`, porque este último se usa para la asignación.

### Reordenar filas

Una y otra vez querrás reordenar las filas de tu dataframe según los valores de una columna concreta. **pandas** lo hace muy fácil con la función `.sort_values()`. Recibe un dataframe y un conjunto de nombres de columnas por los que ordenar. Si proporcionas más de un nombre de columna, cada columna adicional se usará para desempatar los valores de las columnas anteriores. Por ejemplo, el siguiente código ordena por la hora de salida, que está repartida en cuatro columnas.

In [ ]:
flights.sort_values(["year", "month", "day", "dep_time"])

Puedes usar el argumento de palabra clave `ascending=False` para ordenar por una o varias columnas en orden descendente.
Por ejemplo, este código muestra los vuelos con más retraso:

In [ ]:
flights.sort_values("dep_delay", ascending=False)

Por supuesto, puedes combinar todas las manipulaciones de filas anteriores para resolver problemas más complejos.
Por ejemplo, podríamos buscar los tres principales destinos de los vuelos que llegaron con más retraso y que salieron aproximadamente a tiempo:

In [ ]:
(
    flights.query("dep_delay <= 10 and dep_delay >= -10")
    .sort_values("arr_delay", ascending=False)
    .iloc[[0, 1, 2]]
)

### Ejercicios

1. Encuentra todos los vuelos que

    a. Tuvieron un retraso en la llegada de dos horas o más

    b. Volaron a Houston (`"IAH"` o `"HOU"`)

    c. Fueron operados por United, American o Delta

    d. Salieron en verano (julio, agosto y septiembre)

    e. Llegaron con más de dos horas de retraso, pero no salieron tarde

    f. Se retrasaron al menos una hora, pero recuperaron más de 30 minutos en vuelo

2.  Ordena `flights` para encontrar los vuelos con los mayores retrasos en la salida.

3.  Ordena `flights` para encontrar los vuelos más rápidos

4.  ¿Qué vuelos recorrieron la mayor distancia?

5.  ¿Importa el orden en que usas `query()` y `sort_values()` si usas ambas? ¿Por qué sí o por qué no? Piensa en los resultados y en cuánto trabajo tendrían que hacer las funciones.

## Manipulación de columnas

Esta sección te mostrará cómo aplicar a las columnas de tu dataframe las diversas operaciones que puedas necesitar.

::: {.callout-note}
Some **pandas** operations can apply either to columns or rows, depending on the syntax used. For example, accessing values by position can be achieved in the same way for rows and columns via `.iloc` where to access the ith row you would use `df.iloc[i]` and to access the jth column you would use `df.iloc[:, j]` where `:` stands in for 'any row'.
:::

### Crear nuevas columnas

Pasemos ahora a crear nuevas columnas, ya sea con información nueva o a partir de columnas existentes. Dado un dataframe, `df`, crear una nueva columna con el mismo valor repetido es tan fácil como usar corchetes con un string (texto entre comillas) dentro.

In [ ]:
df["new_column0"] = 5
df

Si hacemos la misma operación otra vez, pero con un lado derecho distinto, se sobrescribirá lo que ya había en esa columna. Veámoslo con un ejemplo en el que ponemos valores diferentes en cada posición asignando una lista a la nueva columna.

In [ ]:
df["new_column0"] = [0, 1, 2, 3, 4, 5]
df

::: {.callout-tip title="Exercise"}
What happens if you try to use assignment where the right-hand side values are longer or shorter than the length of the data frame?
:::

Si pasamos una lista dentro de los corchetes, en realidad podemos crear más de una columna nueva:

In [ ]:
df[["new_column1", "new_column2"]] = [5, 6]
df

Muy a menudo querrás crear una nueva columna que sea el resultado de una operación sobre columnas existentes. Hay un par de formas de hacerlo. El método 'independiente' funciona de forma similar a lo que acabamos de ver, salvo que también hacemos referencia al dataframe en el lado derecho de la sentencia de asignación:

In [ ]:
df["new_column3"] = df["col0"] - df["new_column0"]
df

La otra forma de hacerlo implica una sentencia 'assign()' y se usa cuando quieres encadenar varios pasos (como vimos antes). Estas usan una sintaxis especial llamada sentencia 'lambda', que (al menos aquí) simplemente proporciona una forma de indicarle a **pandas** que queremos realizar la operación en cada fila. A continuación tienes un ejemplo con los datos de vuelos. Ten en cuenta, eso sí, que la palabra 'row' de abajo es un marcador; podrías sustituirla por cualquier nombre de variable (por ejemplo, `x`), pero `row` hace que lo que ocurre sea un poco más claro.

In [ ]:
(
    flights.assign(
        gain=lambda row: row["dep_delay"] - row["arr_delay"],
        speed=lambda row: row["distance"] / row["air_time"] * 60,
    )
)

::: {.callout-note}
A lambda function is like any normal function in Python except that it has no name, and it tends to be contained in one line of code. A lambda function is made of an argument, a colon, and an expression, like the following lambda function that multiplies an input by three.

```python
lambda x: x*3
```

:::

### Acceder a columnas



Al igual que al seleccionar filas, hay muchas opciones y formas de seleccionar las columnas sobre las que operar. La de sintaxis más sencilla es el nombre del dataframe seguido de corchetes y el nombre de la columna (como string)

In [ ]:
df["col0"]

Si necesitas seleccionar *varias* columnas, no puedes pasar simplemente un string a `df[...]`; en su lugar, debes pasar un objeto iterable (y que, por tanto, tenga varios elementos). La forma más directa de seleccionar varias columnas es pasar una *lista*. Recuerda que las listas van entre corchetes, así que veremos algo con corchetes repetidos: unos para acceder al interior del dataframe y otros para la lista.

In [ ]:
df[["col0", "new_column0", "col2"]]

Si quieres acceder a filas concretas al mismo tiempo, usa la función de acceso `.loc`:

In [ ]:
df.loc[["a", "b"], ["col0", "new_column0", "col2"]]

Y, al igual que con las filas, podemos acceder a las columnas por su posición usando `.iloc` (donde `:` representa 'cualquier fila').

In [ ]:
df.iloc[:, [0, 1]]

Hay otras formas de acceder a varias columnas que usan slicing (rebanado), pero dejaremos ese tema para otro momento.

A veces querrás seleccionar columnas según el *tipo* de datos que contienen. Para ello, **pandas** proporciona la función `.select_dtypes()`. Usémosla para seleccionar todas las columnas con enteros en los datos de vuelos.

In [ ]:
flights.select_dtypes("int")

En otras ocasiones querrás seleccionar columnas según criterios como patrones en el *nombre* de la columna. Como Python tiene muy buen soporte para texto, esto es muy posible, pero no suele estar tan integrado en las funciones de **pandas**. El truco consiste en generar una lista con los nombres de columna que quieres a partir del patrón que te interesa.

Veamos un par de ejemplos. Primero, obtengamos todas las columnas de nuestro dataframe `df` que empiezan por `"new_..."`. Generaremos una lista de valores verdaderos y falsos que indican si cada columna empieza por "new" y luego pasaremos esos valores a `.loc`, que solo devolverá las columnas cuyo resultado fue `True`. Para mostrar lo que ocurre, lo dividiremos en dos pasos:

In [ ]:
print("The list of columns:")
print(df.columns)
print("\n")

print("The list of true and false values:")
print(df.columns.str.startswith("new"))
print("\n")

print("The selection from the data frame:")
df.loc[:, df.columns.str.startswith("new")]

Además de `startswith()`, hay otros comandos como `endswith()`, `contains()`, `isnumeric()` e `islower()`.

### Renombrar columnas

Hay tres formas sencillas de renombrar columnas, según el contexto. La primera es usar la función específica `rename()` con un objeto llamado diccionario. Los diccionarios en Python constan de llaves con pares de valores separados por comas, donde el primer valor se asigna al segundo. Un ejemplo de diccionario sería `{'old_col1': 'new_col1', 'old_col2': 'new_col2'}`. Veámoslo en la práctica (pero ten en cuenta que no estamos 'guardando' el dataframe resultante, solo mostrándolo; para guardarlo, tendrías que añadir `df = ` al lado izquierdo del código de abajo).

In [ ]:
df.rename(columns={"col3": "letters", "col4": "names", "col6": "fruit"})

El segundo método es para cuando quieres renombrar todas las columnas. Para ello, simplemente igualas `df.columns` al nuevo conjunto de columnas que quieras tener. Por ejemplo, podríamos querer poner en mayúscula la primera letra de cada columna usando `str.capitalize()` y asignarlo a `df.columns`.

In [ ]:
df.columns = df.columns.str.capitalize()
df

Por último, puede que solo nos interese reemplazar partes concretas de los nombres de columna. En este caso, podemos usar `.str.replace()`. Como ejemplo, añadamos la palabra `"Original"` delante de las columnas originales:

In [ ]:
df.columns.str.replace("Col", "Original_column")

### Reordenar columnas

Por defecto, las nuevas columnas se añaden al lado derecho del dataframe. Pero puedes tener motivos para querer que las columnas aparezcan en un orden concreto, o quizá simplemente te resulte más cómodo tener las nuevas columnas a la izquierda cuando hay muchas columnas en un dataframe (lo cual pasa mucho).

La forma más sencilla de reordenar (todas) las columnas es crear una nueva lista con sus nombres en el orden que quieras: ¡pero ten cuidado de no olvidar ninguna columna que quieras conservar! 

Veamos un ejemplo con una versión nueva de los datos ficticios de antes. Pondremos primero todas las columnas impares, en orden descendente, y luego las pares de la misma forma.

In [ ]:
df = pd.DataFrame(
    data=np.reshape(range(36), (6, 6)),
    index=["a", "b", "c", "d", "e", "f"],
    columns=["col" + str(i) for i in range(6)],
    dtype=float,
)
df

In [ ]:
df = df[["col5", "col3", "col1", "col4", "col2", "col0"]]
df

¡Por supuesto, esto es bastante tedioso si tienes muchas columnas! Hay métodos que pueden facilitarlo según tu contexto. ¿Quizá solo quieras ordenar las columnas? Esto se consigue combinando `sorted()` y el comando `reindex()` (que funciona para filas o columnas) con `axis=1`, que indica el segundo eje (es decir, las columnas).

In [ ]:
df.reindex(sorted(df.columns), axis=1)

## Repaso de cómo acceder a filas, columnas y valores

Con tantas formas distintas de acceder a valores en los dataframes, es fácil confundirse. Estas son las distintas formas de obtener la primera columna de un dataframe (cuando esa primera columna se llama `column` y el dataframe es `df`):

- `df.column`
- `df["column"]`
- `df.loc[:, "column"]`
- `df.iloc[:, 0]`

¡Ten en cuenta que `:` significa 'dame todo'! Las formas de acceder a las filas son similares (suponiendo aquí que la primera fila se llama `row`):

- `df.loc["row", :]`
- `df.iloc[0, :]`

Y para acceder al primer valor (es decir, el valor de la primera fila y la primera columna):

- `df.column[0]`
- `df["column"][0]`
- `df.iloc[0, 0]`
- `df.loc["row", "column"]`

En los ejemplos anteriores, los corchetes son instrucciones sobre *dónde* tomar fragmentos del dataframe. Son algo así como un sistema de direcciones para los valores dentro de un dataframe. Sin embargo, los corchetes *también* denotan listas. Así que si quieres seleccionar *varias* columnas o filas, puede que veas una sintaxis como esta:

`df.loc[["row0", "row1"], ["column0", "column2"]]`

que selecciona dos filas y dos columnas mediante las listas `["row0", "row1"]` y `["column0", "column2"]`. Como hay listas junto al sistema habitual de selección de valores, aparecen dos pares de corchetes.

::: {.callout-tip title="Tip"}

Si solo quieres recordar una sintaxis para acceder a filas y columnas por nombre, usa el patrón `df.loc[["row0", "row1", ...], ["col0", "col1", ...]]`. También funciona con una sola fila o una sola columna (o ambas).

Si solo quieres recordar una sintaxis para acceder a filas y columnas por posición, usa el patrón `df.iloc[[0, 1, ...], [0, 1, ...]]`. También funciona con una sola fila o una sola columna (o ambas).
:::


### Ejercicios de columnas y filas

1.  Compara `air_time` con `arr_time - dep_time`. ¿Qué esperas ver? ¿Qué ves? ¿Qué necesitas hacer para corregirlo?

2.  Compara `dep_time`, `sched_dep_time` y `dep_delay`. ¿Cómo esperarías que se relacionen esos tres números?

3.  Piensa en tantas formas como sea posible de seleccionar `dep_time`, `dep_delay`, `arr_time` y `arr_delay` de `flights`.

4.  ¿Qué ocurre si incluyes el nombre de una fila o columna varias veces al intentar seleccionarlas?

5.  ¿Qué hace la función `.isin()` en el siguiente código?

    ```python
    flights.columns.isin(["year", "month", "day", "dep_delay", "arr_delay"])
    ```

6.  ¿Te sorprende el resultado de ejecutar el siguiente código?
    ¿Cómo tratan por defecto las mayúsculas y minúsculas funciones como `str.contains`?
    ¿Cómo puedes cambiar ese comportamiento por defecto?

    ```python
    flights.loc[:, flights.columns.str.contains("TIME")]
    ```

    (Pista: puedes usar la ayuda incluso con funciones que se aplican a dataframes, por ejemplo, usa `help(flights.columns.str.contains)`)

## Agrupar, cambiar el índice y aplicar estadísticos de resumen

Hasta ahora has aprendido a trabajar con filas y columnas. **pandas** se vuelve aún más potente cuando añades la capacidad de trabajar con grupos. Crear grupos suele implicar también un cambio de índice. Y como los grupos tienden a implicar una agregación o combinación de datos, a menudo van de la mano de la aplicación de un estadístico de resumen.

El siguiente diagrama da una idea de cómo estas operaciones pueden realizarse en conjunto. Observa que la operación de 'dividir' se logra mediante la agrupación, mientras que la de aplicar produce estadísticas de resumen. Al final, obtienes un dataframe con un nuevo índice (una entrada por grupo) en lo que se muestra como el paso de 'combinar'.

![](https://jakevdp.github.io/PythonDataScienceHandbook/figures/03.08-split-apply-combine.png)

### Agrupar y agregar

Veamos cómo crear un grupo con la función `.groupby()`, seguida de la selección de una columna y la aplicación de una estadística de resumen mediante una agregación. Observa que la *agregación*, mediante `.agg()`, siempre produce un nuevo índice porque hemos condensado la información al nivel de grupo (y el nuevo índice está formado por esos niveles).

El punto clave que debes recordar es: usa `.agg()` con `.groupby()` cuando quieras que tus grupos se conviertan en el nuevo índice.

In [ ]:
(flights.groupby("month")[["dep_delay"]].mean())

Esto representa ahora el retraso medio de salida por mes. ¡Observa que nuestro índice ha cambiado! Ahora tenemos el mes donde originalmente teníamos un índice que era solo el número de fila. El índice desempeña un papel importante en las operaciones de agrupación porque lleva el registro de los grupos que tienes en el resto de tu dataframe.

A menudo querrás hacer varias operaciones de resumen de una sola vez. La sintaxis más completa para esto es mediante `.agg()`. Podemos reproducir lo que hicimos antes usando `.agg()`:

In [ ]:
(flights.groupby("month")[["dep_delay"]].agg("mean"))

donde pasas la agregación que quieras. Algunas opciones comunes están en la siguiente tabla:

| Agregación      | Descripción |
| ----------- | ----------- |
| `count()`      | Número de elementos       |
| `first()`, `last()` | 	Primer y último elemento |
| `mean()`, `median()` |	Media y mediana |
| `min()`, `max()` |	Mínimo y máximo |
| `std()`, `var()` |	Desviación estándar y varianza |
| `mad()` |	Desviación absoluta media |
| `prod()` |	Producto de todos los elementos |
| `sum()`	| Suma de todos los elementos |
| `value_counts()` | Conteo de valores únicos |

Para hacer varias agregaciones sobre varias columnas con nuevos nombres para las variables de salida, la sintaxis queda así

In [ ]:
(
    flights.groupby(["month"]).agg(
        mean_delay=("dep_delay", "mean"),
        count_flights=("dep_delay", "count"),
    )
)

¡Las medias y los conteos te pueden llevar sorprendentemente lejos en ciencia de datos!

### Agrupar por varias variables

Esto es tan sencillo como pasarle a `.groupby()` una lista que represente varias columnas en lugar de un string que represente una sola columna.

In [ ]:
month_year_delay = flights.groupby(["month", "year"]).agg(
    mean_delay=("dep_delay", "mean"),
    count_flights=("dep_delay", "count"),
)
month_year_delay

Quizá hayas notado que esta vez tenemos un multi-índice (es decir, un índice con más de una columna). Eso se debe a que pedimos algo con varios grupos, y el índice registra lo que ocurre dentro de cada grupo: por eso necesitamos más de una dimensión de índice para hacerlo de forma eficiente.

Si alguna vez quieres volver a un índice que sea solo la posición, prueba `reset_index()`

In [ ]:
month_year_delay.reset_index()

Aunque quizá solo quieras eliminar un nivel del índice. Esto se consigue pasando la posición del índice que quieres eliminar: por ejemplo, para convertir solo el índice del año en una columna, usaríamos: 

In [ ]:
month_year_delay.reset_index(1)

Por último, puedes hacer reorganizaciones más complicadas del índice con una operación llamada `unstack`, que pivota la variable de índice elegida para que pase a ser una variable de columna (lo que introduce una estructura de columnas con varios niveles). Normalmente es mejor evitarlo.

### Agrupar y transformar

Puede que no siempre quieras cambiar el índice para reflejar los nuevos grupos al realizar cálculos a nivel de grupo.

El punto clave que debes recordar es: usa `.transform()` con `.groupby()` cuando quieras realizar cálculos sobre tus grupos pero quieras volver al índice original.

Supongamos que queremos expresar el retraso de llegada, `"arr_del"`, de cada vuelo como una fracción del peor retraso de llegada de cada mes.

In [ ]:
flights["max_delay_month"] = flights.groupby("month")["arr_delay"].transform("max")
flights["delay_frac_of_max"] = flights["arr_delay"] / flights["max_delay_month"]
flights[
    ["year", "month", "day", "arr_delay", "max_delay_month", "delay_frac_of_max"]
].head()

Observa que las primeras entradas de `"max_delay_month"` son todas iguales porque el mes es el mismo para esas entradas, pero la fracción de retraso cambia en cada fila.

### Ejercicios de groupby

1.  ¿Qué aerolínea tiene los peores retrasos? Desafío: ¿puedes separar los efectos de los malos aeropuertos frente a los de las malas aerolíneas? ¿Por qué sí o por qué no? (Pista: piensa en `flights.groupby(["carrier", "dest"]).count()`)

2.  Encuentra el vuelo con más retraso para cada destino.

3.  ¿Cómo varían los retrasos a lo largo del día?